In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = "/content/drive/MyDrive"

In [ ]:
import os
import json
import pandas as pd
from sklearn.model_selection import train_test_split

BASE = "/content/drive/MyDrive/SinhThanh"

TRAIN_ROOT = "/content/drive/MyDrive/SinhThanh/Data/MDD-Challenge-2025-training-set"
TEST_ROOT  = "/content/drive/MyDrive/SinhThanh/Data/MDD-Challenge-2025-public-test"

TRAIN_META = f"{TRAIN_ROOT}/metadata/train_phones.csv"
PUBLIC_META = f"{TEST_ROOT}/metadata/public_test_phones.csv"

TRAIN_WAV_DIR = TRAIN_ROOT
PUBLIC_WAV_DIR = TEST_ROOT

WORK_DIR = "/content/drive/MyDrive/SinhThanh/MDD_Work"
os.makedirs(WORK_DIR, exist_ok=True)

print("TRAIN_ROOT exists:", os.path.exists(TRAIN_ROOT))
print("TEST_ROOT exists:", os.path.exists(TEST_ROOT))
print("TRAIN_META exists:", os.path.exists(TRAIN_META))
print("PUBLIC_META exists:", os.path.exists(PUBLIC_META))
print("WORK_DIR:", WORK_DIR)

print("\nTrain metadata:")
print(os.listdir(f"{TRAIN_ROOT}/metadata"))

print("\nPublic metadata:")
print(os.listdir(f"{TEST_ROOT}/metadata"))

In [ ]:
!pip uninstall -y numpy scipy pandas scikit-learn librosa numba llvmlite soundfile soxr transformers

!pip install -q --no-cache-dir --force-reinstall \
    numpy==1.26.4 \
    scipy==1.13.1 \
    pandas==2.2.2 \
    scikit-learn==1.5.2 \
    librosa==0.10.2.post1 \
    soundfile==0.12.1 \
    soxr==0.3.7 \
    numba==0.60.0 \
    llvmlite==0.43.0

!pip install -q --no-cache-dir --force-reinstall \
    transformers==4.40.2 \
    jiwer \
    pyctcdecode \
    tqdm

print("Install done. BẮT BUỘC restart runtime/session rồi chạy lại từ đầu.")

In [ ]:
import torch
import transformers
import librosa
import pandas as pd
import numpy as np
import sklearn
import scipy
import soundfile
import soxr

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("librosa:", librosa.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("soundfile:", soundfile.__version__)
print("soxr:", soxr.__version__)

from scipy.signal import resample_poly
print("scipy.signal OK")

assert torch.cuda.is_available(), "CUDA đang False, bật GPU đi bro"
assert np.__version__.startswith("1.26"), "numpy chưa đúng 1.26.x"
assert scipy.__version__.startswith("1.13"), "scipy chưa đúng 1.13.x"
assert librosa.__version__ == "0.10.2.post1", "librosa chưa đúng version"

print("Environment OK.")

In [ ]:
train_df = pd.read_csv(TRAIN_META)
public_df = pd.read_csv(PUBLIC_META)

print("Train shape:", train_df.shape)
print("Public shape:", public_df.shape)

print("\nTrain head:")
print(train_df.head())

print("\nPublic head:")
print(public_df.head())

print("\nTrain columns:", train_df.columns.tolist())
print("Public columns:", public_df.columns.tolist())

In [ ]:
def check_audio(df, root, n=5):
    for i in range(n):
        p = df.iloc[i]["path"]
        full_path = os.path.join(root, p)
        print(full_path, "=>", os.path.exists(full_path))

print("Train audio check:")
check_audio(train_df, TRAIN_ROOT, 5)

print("\nPublic audio check:")
check_audio(public_df, TEST_ROOT, 5)

In [ ]:
SPECIAL_BLANK = "<blank>"
SPECIAL_UNK = "<unk>"

tokens = set()

for col in ["canonical", "transcript"]:
    for sent in train_df[col].astype(str).tolist():
        for tok in sent.split():
            if tok and tok != "$":
                tokens.add(tok)

tokens = sorted(tokens)

vocab = {SPECIAL_BLANK: 0, SPECIAL_UNK: 1}
for tok in tokens:
    if tok not in vocab:
        vocab[tok] = len(vocab)

vocab_path = f"{WORK_DIR}/vocab.json"
with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

BLANK_ID = vocab[SPECIAL_BLANK]
UNK_ID = vocab[SPECIAL_UNK]
id2token = {v: k for k, v in vocab.items()}
vocab_size = len(vocab)

print("Vocab size:", vocab_size)
print(list(vocab.items())[:30])
print("Saved:", vocab_path)
print("BLANK_ID:", BLANK_ID)
print("UNK_ID:", UNK_ID)


In [ ]:
train_split, dev_split = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

train_split_path = f"{WORK_DIR}/train_split.csv"
dev_split_path = f"{WORK_DIR}/dev_split.csv"
public_gt_path = f"{WORK_DIR}/public_gt.csv"

train_split.to_csv(train_split_path, index=False)
dev_split.to_csv(dev_split_path, index=False)
public_df.to_csv(public_gt_path, index=False)

print("Train split:", train_split.shape, train_split_path)
print("Dev split:", dev_split.shape, dev_split_path)
print("Public GT:", public_df.shape, public_gt_path)

In [ ]:
import os, json, random, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa

from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoFeatureExtractor
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_RATE = 16000

PRETRAINED_MODEL = "nguyenvulebinh/wav2vec2-base-vietnamese-250h"

BATCH_SIZE = 4
EPOCHS = 15
LR = 1e-5
EVAL_START_EPOCH = 2

BLANK_ID = 0
UNK_ID = 1

print("Device:", DEVICE)

In [ ]:
with open(vocab_path, "r", encoding="utf-8") as f:
    vocab = json.load(f)

id2token = {v: k for k, v in vocab.items()}
vocab_size = len(vocab)

def text_to_ids(text):
    ids = []
    for tok in str(text).split():
        if tok and tok != "$":
            ids.append(vocab.get(tok, UNK_ID))
    return ids

def greedy_decode(logits):
    """
    logits: Tensor [T, V]
    """
    pred_ids = torch.argmax(logits, dim=-1).detach().cpu().tolist()
    collapsed = []
    prev = None

    for idx in pred_ids:
        if idx != prev:
            collapsed.append(idx)
        prev = idx

    tokens = []
    for idx in collapsed:
        if idx == BLANK_ID:
            continue
        tok = id2token.get(idx, "")
        if tok not in ["<blank>", "<unk>", "$", ""]:
            tokens.append(tok)
    return " ".join(tokens)

print("Vocab size:", vocab_size)
print(list(vocab.items())[:20])
print("BLANK_ID:", BLANK_ID, "=>", id2token[BLANK_ID])
print("UNK_ID:", UNK_ID, "=>", id2token[UNK_ID])

In [ ]:
class MDDPhoneDataset(Dataset):
    def __init__(self, csv_path, root_dir):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.root_dir = root_dir

    def __len__(self):
        return len(self.df)

    def _resolve_path(self, p):
        p = str(p)
        if os.path.isabs(p):
            return p
        return os.path.join(self.root_dir, p)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav_path = self._resolve_path(row["path"])

        wav, _ = librosa.load(wav_path, sr=SAMPLE_RATE)

        canonical_ids = text_to_ids(row["canonical"])
        transcript_ids = text_to_ids(row["transcript"])

        return {
            "wav": wav,
            "canonical_ids": canonical_ids,
            "transcript_ids": transcript_ids,
            "canonical": row["canonical"],
            "transcript": row["transcript"],
            "path": row["path"],
        }


feature_extractor = AutoFeatureExtractor.from_pretrained(PRETRAINED_MODEL)

def collate_fn(batch):
    wavs = [b["wav"] for b in batch]

    inputs = feature_extractor(
        wavs,
        sampling_rate=SAMPLE_RATE,
        padding=True,
        return_tensors="pt"
    )

    input_values = inputs.input_values

    max_can_len = max(len(b["canonical_ids"]) for b in batch)
    max_lab_len = max(len(b["transcript_ids"]) for b in batch)

    canonical = torch.full((len(batch), max_can_len), BLANK_ID, dtype=torch.long)
    labels = torch.full((len(batch), max_lab_len), BLANK_ID, dtype=torch.long)
    label_lengths = torch.zeros(len(batch), dtype=torch.long)

    for i, b in enumerate(batch):
        can = b["canonical_ids"]
        lab = b["transcript_ids"]

        canonical[i, :len(can)] = torch.tensor(can, dtype=torch.long)
        labels[i, :len(lab)] = torch.tensor(lab, dtype=torch.long)
        label_lengths[i] = len(lab)

    return {
        "input_values": input_values,
        "canonical": canonical,
        "labels": labels,
        "label_lengths": label_lengths,
        "transcripts": [b["transcript"] for b in batch],
        "canonicals": [b["canonical"] for b in batch],
        "paths": [b["path"] for b in batch],
    }

In [ ]:
train_ds = MDDPhoneDataset(train_split_path, TRAIN_ROOT)
dev_ds = MDDPhoneDataset(dev_split_path, TRAIN_ROOT)
public_ds = MDDPhoneDataset(public_gt_path, TEST_ROOT)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0
)

dev_loader = DataLoader(
    dev_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

public_loader = DataLoader(
    public_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

print("Train batches:", len(train_loader))
print("Dev samples:", len(dev_ds))
print("Public samples:", len(public_ds))

In [ ]:
class AcousticLinguisticMDD(nn.Module):
    def __init__(self, pretrained_name, vocab_size):
        super().__init__()

        self.audio_encoder = AutoModel.from_pretrained(pretrained_name)
        hidden = self.audio_encoder.config.hidden_size

        # Freeze feature extractor cho ổn định khi data ít
        if hasattr(self.audio_encoder, "feature_extractor"):
            for p in self.audio_encoder.feature_extractor.parameters():
                p.requires_grad = False

        self.phone_embedding = nn.Embedding(
            vocab_size,
            128,
            padding_idx=BLANK_ID
        )

        self.phone_encoder = nn.LSTM(
            input_size=128,
            hidden_size=hidden // 2,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden,
            num_heads=8,
            dropout=0.1,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, vocab_size)
        )

    def forward(self, input_values, canonical):
        audio_out = self.audio_encoder(input_values).last_hidden_state
        # audio_out: [B, T, H]

        phone_emb = self.phone_embedding(canonical)
        phone_out, _ = self.phone_encoder(phone_emb)
        # phone_out: [B, L, H]

        attn_out, _ = self.cross_attn(
            query=audio_out,
            key=phone_out,
            value=phone_out
        )

        fused = torch.cat([audio_out, attn_out], dim=-1)
        fused = self.dropout(fused)

        logits = self.classifier(fused)
        return logits


model = AcousticLinguisticMDD(PRETRAINED_MODEL, vocab_size).to(DEVICE)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total params:", total)
print("Trainable params:", trainable)
print("Device:", DEVICE)

In [ ]:
def _align(seq1, seq2):
    GAP = -1
    MATCH = 1
    MISMATCH = -1

    n, m = len(seq1), len(seq2)
    score = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        score[i][0] = GAP * i
    for j in range(n + 1):
        score[0][j] = GAP * j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if seq1[j - 1] == seq2[i - 1]:
                s = MATCH
            elif seq1[j - 1] == "<eps>" or seq2[i - 1] == "<eps>":
                s = GAP
            else:
                s = MISMATCH

            score[i][j] = max(
                score[i - 1][j - 1] + s,
                score[i - 1][j] + GAP,
                score[i][j - 1] + GAP,
            )

    align1, align2 = [], []
    i, j = m, n

    while i > 0 and j > 0:
        if seq1[j - 1] == seq2[i - 1]:
            s = MATCH
        elif seq1[j - 1] == "<eps>" or seq2[i - 1] == "<eps>":
            s = GAP
        else:
            s = MISMATCH

        if score[i][j] == score[i - 1][j - 1] + s:
            align1.append(seq1[j - 1])
            align2.append(seq2[i - 1])
            i -= 1
            j -= 1
        elif score[i][j] == score[i][j - 1] + GAP:
            align1.append(seq1[j - 1])
            align2.append("<eps>")
            j -= 1
        else:
            align1.append("<eps>")
            align2.append(seq2[i - 1])
            i -= 1

    while j > 0:
        align1.append(seq1[j - 1])
        align2.append("<eps>")
        j -= 1

    while i > 0:
        align1.append("<eps>")
        align2.append(seq2[i - 1])
        i -= 1

    align1.reverse()
    align2.reverse()
    return align1, align2


def _ops(a1, a2):
    ops = []
    for r, h in zip(a1, a2):
        if r != "<eps>" and h == "<eps>":
            ops.append("D")
        elif r == "<eps>" and h != "<eps>":
            ops.append("I")
        elif r != h:
            ops.append("S")
        else:
            ops.append("C")
    return ops


def _align_pair(s1, s2):
    seq1 = str(s1).replace("*", "").split()
    seq2 = str(s2).replace("*", "").split()
    a1, a2 = _align(seq1, seq2)
    return a1, a2, _ops(a1, a2)


def compute_mdd_score(gt_df, pred_list):
    total_per_errors = 0
    total_phonemes = 0

    cor_nocor = 0
    sub_sub = sub_sub1 = sub_nosub = 0
    ins_ins = ins_ins1 = ins_noins = 0
    del_del = del_del1 = del_nodel = 0

    for i, row in gt_df.reset_index(drop=True).iterrows():
        ref_str = row["canonical"]
        human_str = row["transcript"]
        our_str = pred_list[i]

        human_tokens = str(human_str).replace("*", "").split()
        predict_tokens = str(our_str).replace("*", "").split()

        h_al, p_al, op_hp = _align_pair(
            " ".join(human_tokens),
            " ".join(predict_tokens)
        )

        total_per_errors += sum(1 for op in op_hp if op in ("I", "D", "S"))
        total_phonemes += len(human_tokens)

        ref_seq, human_seq, op_rh = _align_pair(ref_str, human_str)
        human_seq2, our_seq2, op_ho = _align_pair(human_str, our_str)
        ref_seq3, our_seq3, op_ro = _align_pair(ref_str, our_str)

        flag = 0
        for j in range(len(ref_seq)):
            if ref_seq[j] == "<eps>":
                continue
            while flag < len(ref_seq3) and ref_seq3[flag] == "<eps>":
                flag += 1
            if flag < len(ref_seq3) and ref_seq[j] == ref_seq3[flag]:
                if op_rh[j] == "D" and op_ro[flag] == "D":
                    del_del += 1
                elif op_rh[j] == "D" and op_ro[flag] != "D" and op_ro[flag] != "C":
                    del_del1 += 1
                elif op_rh[j] == "D" and op_ro[flag] != "D" and op_ro[flag] == "C":
                    del_nodel += 1
                flag += 1

        flag = 0
        for j in range(len(human_seq)):
            if human_seq[j] == "<eps>":
                continue
            while flag < len(human_seq2) and human_seq2[flag] == "<eps>":
                flag += 1

            if flag < len(human_seq2) and human_seq[j] == human_seq2[flag]:
                if op_rh[j] == "C" and op_ho[flag] != "C":
                    cor_nocor += 1

                if op_rh[j] == "S" and op_ho[flag] == "C":
                    sub_sub += 1
                elif op_rh[j] == "S" and op_ho[flag] != "C" and ref_seq[j] != our_seq2[flag]:
                    sub_sub1 += 1
                elif op_rh[j] == "S" and op_ho[flag] != "C" and ref_seq[j] == our_seq2[flag]:
                    sub_nosub += 1

                if op_rh[j] == "I" and op_ho[flag] == "C":
                    ins_ins += 1
                elif op_rh[j] == "I" and op_ho[flag] != "C" and op_ho[flag] != "D":
                    ins_ins1 += 1
                elif op_rh[j] == "I" and op_ho[flag] != "C" and op_ho[flag] == "D":
                    ins_noins += 1

                flag += 1

    PER = total_per_errors / total_phonemes if total_phonemes > 0 else 0.0

    TR = sub_sub + sub_sub1 + del_del + del_del1 + ins_ins + ins_ins1
    FR = cor_nocor
    FA = sub_nosub + ins_noins + del_nodel

    precision = TR / (TR + FR) if (TR + FR) > 0 else 0.0
    recall = TR / (TR + FA) if (TR + FA) > 0 else 0.0
    F1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    correct_diag = sub_sub + ins_ins + del_del
    error_diag = sub_sub1 + ins_ins1 + del_del1
    DER = error_diag / (correct_diag + error_diag) if (correct_diag + error_diag) > 0 else 0.0

    Score = 0.5 * F1 + 0.4 * (1 - DER) + 0.1 * (1 - PER)

    return Score, F1, DER, PER


print("Metric functions loaded.")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
ctc_loss = nn.CTCLoss(blank=BLANK_ID, zero_infinity=True)

def train_one_epoch(epoch):
    model.train()
    losses = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    for batch in pbar:
        input_values = batch["input_values"].to(DEVICE)
        canonical = batch["canonical"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        label_lengths = batch["label_lengths"].to(DEVICE)

        logits = model(input_values, canonical)   # [B, T, V]
        log_probs = F.log_softmax(logits, dim=-1)

        B, T, V = log_probs.shape
        input_lengths = torch.full(
            size=(B,),
            fill_value=T,
            dtype=torch.long,
            device=DEVICE
        )

        loss = ctc_loss(
            log_probs.transpose(0, 1),
            labels,
            input_lengths,
            label_lengths
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        losses.append(loss.item())
        pbar.set_postfix(loss=np.mean(losses))

    return float(np.mean(losses))


@torch.no_grad()
def predict_loader(loader):
    model.eval()
    preds = []

    for batch in tqdm(loader, desc="Predict"):
        input_values = batch["input_values"].to(DEVICE)
        canonical = batch["canonical"].to(DEVICE)

        logits = model(input_values, canonical)
        log_probs = F.log_softmax(logits, dim=-1)

        for i in range(log_probs.shape[0]):
            pred = greedy_decode(log_probs[i])
            preds.append(pred)

    return preds


def evaluate_on_csv(loader, csv_path):
    gt_df = pd.read_csv(csv_path).reset_index(drop=True)
    preds = predict_loader(loader)
    score, f1, der, per = compute_mdd_score(gt_df, preds)
    return score, f1, der, per, preds


print("Train/eval functions loaded.")

In [ ]:
CKPT_DIR = f"{WORK_DIR}/checkpoints_new_account"
os.makedirs(CKPT_DIR, exist_ok=True)

best_score = -1.0
best_path = f"{CKPT_DIR}/best_mdd_vietnamese.pt"
last_path = f"{CKPT_DIR}/last_mdd_vietnamese.pt"
history_path = f"{WORK_DIR}/history_new_account.csv"

history = []

print("CKPT_DIR:", CKPT_DIR)
print("best_path:", best_path)
print("last_path:", last_path)
print("history_path:", history_path)

print("\nStart CLEAN training from scratch...")
print(f"Training from epoch 1 to {EPOCHS}")
print(f"Initial best_score: {best_score:.6f}")


for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(epoch)
    print(f"\nEpoch {epoch} | Train loss: {loss:.4f}")

    # Save last checkpoint nhẹ, không lưu optimizer
    torch.save({
        "model": model.state_dict(),
        "vocab": vocab,
        "epoch": epoch,
        "loss": float(loss),
        "best_score": float(best_score),
        "config": {
            "pretrained": PRETRAINED_MODEL,
            "vocab_size": vocab_size
        }
    }, last_path)

    print("Saved last checkpoint:", last_path)

    if epoch >= EVAL_START_EPOCH:
        score, f1, der, per, _ = evaluate_on_csv(dev_loader, dev_split_path)

        score = float(score)
        f1 = float(f1)
        der = float(der)
        per = float(per)

        print(
            f"DEV | Score={score:.6f} | F1={f1:.6f} | "
            f"DER={der:.6f} | PER={per:.6f}"
        )

        history.append({
            "epoch": epoch,
            "loss": float(loss),
            "score": score,
            "f1": f1,
            "der": der,
            "per": per
        })

        pd.DataFrame(history).to_csv(history_path, index=False)
        print("Saved history:", history_path)

        if score > best_score:
            best_score = score

            # Save best checkpoint nhẹ, không lưu optimizer
            torch.save({
                "model": model.state_dict(),
                "vocab": vocab,
                "epoch": epoch,
                "score": score,
                "loss": float(loss),
                "f1": f1,
                "der": der,
                "per": per,
                "config": {
                    "pretrained": PRETRAINED_MODEL,
                    "vocab_size": vocab_size
                }
            }, best_path)

            print("Saved best checkpoint:", best_path)

print("\nBest DEV score:", best_score)
pd.DataFrame(history)

TestData


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa

from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoFeatureExtractor
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_RATE = 16000

BASE = "/content/drive/MyDrive/SinhThanh"

TEST_ROOT = f"{BASE}/Data/MDD-Challenge-2025-public-test"
PUBLIC_META = f"{TEST_ROOT}/metadata/public_test_phones.csv"

WORK_DIR = f"{BASE}/MDD_Work"
CKPT_PATH = f"{WORK_DIR}/checkpoints_new_account/best_mdd_vietnamese.pt"

print("DEVICE:", DEVICE)
print("PUBLIC_META exists:", os.path.exists(PUBLIC_META), PUBLIC_META)
print("CKPT exists:", os.path.exists(CKPT_PATH), CKPT_PATH)

assert os.path.exists(PUBLIC_META), "Không thấy public_test_phones.csv"
assert os.path.exists(CKPT_PATH), "Không thấy best checkpoint"

In [ ]:
# ============================================================
# LOAD CHECKPOINT + DEFINE MODEL + PUBLIC LOADER
# ============================================================

import os
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa

from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoFeatureExtractor
from tqdm.auto import tqdm

# Load checkpoint trước
checkpoint = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)

vocab = checkpoint["vocab"]
id2token = {v: k for k, v in vocab.items()}
vocab_size = len(vocab)

BLANK_ID = vocab["<blank>"]
UNK_ID = vocab["<unk>"]
PRETRAINED_MODEL = checkpoint["config"]["pretrained"]

print("Loaded checkpoint:", CKPT_PATH)
print("Best epoch:", checkpoint.get("epoch"))
print("DEV Score:", checkpoint.get("score"))
print("DEV F1:", checkpoint.get("f1"))
print("DEV DER:", checkpoint.get("der"))
print("DEV PER:", checkpoint.get("per"))
print("Vocab size:", vocab_size)
print("PRETRAINED_MODEL:", PRETRAINED_MODEL)


def text_to_ids(text):
    ids = []
    for tok in str(text).split():
        if tok and tok != "$":
            ids.append(vocab.get(tok, UNK_ID))
    return ids


def greedy_decode(logits):
    pred_ids = torch.argmax(logits, dim=-1).detach().cpu().tolist()

    collapsed = []
    prev = None

    for idx in pred_ids:
        if idx != prev:
            collapsed.append(idx)
        prev = idx

    tokens = []
    for idx in collapsed:
        if idx == BLANK_ID:
            continue
        tok = id2token.get(idx, "")
        if tok not in ["<blank>", "<unk>", "$", ""]:
            tokens.append(tok)

    return " ".join(tokens)


class AcousticLinguisticMDD(nn.Module):
    def __init__(self, pretrained_name, vocab_size):
        super().__init__()

        self.audio_encoder = AutoModel.from_pretrained(pretrained_name)
        hidden = self.audio_encoder.config.hidden_size

        if hasattr(self.audio_encoder, "feature_extractor"):
            for p in self.audio_encoder.feature_extractor.parameters():
                p.requires_grad = False

        self.phone_embedding = nn.Embedding(
            vocab_size,
            128,
            padding_idx=BLANK_ID
        )

        self.phone_encoder = nn.LSTM(
            input_size=128,
            hidden_size=hidden // 2,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden,
            num_heads=8,
            dropout=0.1,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, vocab_size)
        )

    def forward(self, input_values, canonical):
        audio_out = self.audio_encoder(input_values).last_hidden_state

        phone_emb = self.phone_embedding(canonical)
        phone_out, _ = self.phone_encoder(phone_emb)

        attn_out, _ = self.cross_attn(
            query=audio_out,
            key=phone_out,
            value=phone_out
        )

        fused = torch.cat([audio_out, attn_out], dim=-1)
        fused = self.dropout(fused)

        logits = self.classifier(fused)
        return logits


# Load model
model = AcousticLinguisticMDD(PRETRAINED_MODEL, vocab_size).to(DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()

print("Model loaded.")


# Public loader
feature_extractor = AutoFeatureExtractor.from_pretrained(PRETRAINED_MODEL)

public_df = pd.read_csv(PUBLIC_META).reset_index(drop=True)


class PublicDataset(Dataset):
    def __init__(self, csv_path, root_dir):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.root_dir = root_dir

    def __len__(self):
        return len(self.df)

    def _resolve_path(self, p):
        p = str(p)
        if os.path.isabs(p):
            return p
        return os.path.join(self.root_dir, p)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav_path = self._resolve_path(row["path"])

        wav, _ = librosa.load(wav_path, sr=SAMPLE_RATE)
        canonical_ids = text_to_ids(row["canonical"])

        return {
            "wav": wav,
            "canonical_ids": canonical_ids,
        }


def collate_fn(batch):
    wavs = [b["wav"] for b in batch]

    inputs = feature_extractor(
        wavs,
        sampling_rate=SAMPLE_RATE,
        padding=True,
        return_tensors="pt"
    )

    max_can_len = max(len(b["canonical_ids"]) for b in batch)
    canonical = torch.full((len(batch), max_can_len), BLANK_ID, dtype=torch.long)

    for i, b in enumerate(batch):
        can = b["canonical_ids"]
        canonical[i, :len(can)] = torch.tensor(can, dtype=torch.long)

    return {
        "input_values": inputs.input_values,
        "canonical": canonical,
    }


public_ds = PublicDataset(PUBLIC_META, TEST_ROOT)

public_loader = DataLoader(
    public_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

print("Public samples:", len(public_ds))
print("Public loader ready.")

In [ ]:
# ============================================================
# CELL 3: PREDICT PUBLIC + PRINT SCORE FOR REPORT
# ============================================================

assert "compute_mdd_score" in globals(), "Chưa có compute_mdd_score. Chạy lại cell metric trước bro."
assert "model" in globals(), "Chưa load model. Chạy Cell 2 trước."
assert "public_loader" in globals(), "Chưa có public_loader. Chạy Cell 2 trước."
assert "public_df" in globals(), "Chưa có public_df. Chạy Cell 2 trước."

@torch.inference_mode()
def predict_public(loader):
    model.eval()
    preds = []

    for batch in tqdm(loader, desc="Predict public"):
        input_values = batch["input_values"].to(DEVICE)
        canonical = batch["canonical"].to(DEVICE)

        logits = model(input_values, canonical)
        log_probs = F.log_softmax(logits, dim=-1)

        pred = greedy_decode(log_probs[0])
        preds.append(pred)

        del input_values, canonical, logits, log_probs

    return preds


public_preds = predict_public(public_loader)

public_score, public_f1, public_der, public_per = compute_mdd_score(
    public_df,
    public_preds
)

print("\n===== RESULT FOR REPORT =====")
print("Checkpoint:", CKPT_PATH)
print("Best DEV epoch:", checkpoint.get("epoch"))
print("Best DEV Score:", checkpoint.get("score"))
print("Best DEV F1:", checkpoint.get("f1"))
print("Best DEV DER:", checkpoint.get("der"))
print("Best DEV PER:", checkpoint.get("per"))

print(
    f"PUBLIC | Score={public_score:.6f} | F1={public_f1:.6f} | "
    f"DER={public_der:.6f} | PER={public_per:.6f}"
)

print("\nFirst 10 predictions:")
for i in range(min(10, len(public_preds))):
    print(i, "=>", public_preds[i])